# Mejoras de trabajo final

A continuacion se detallaran las mejoras y correcciones propuestas en la correccion enviada por el profesor de la catedra:

---

## 1. MLOps y Código

Si bien en el trabajo final pudimos medir el RAG realizado en el MVP, es necesario seguir midiendo distintas funcionalidades de nuestro agente. A continuación planteamos cómo testeamos el uso de las herramientas del agente.

### 1.1 Tests automáticos

La suite quedó organizada por tema en tres archivos dentro de `tests/`:

- `tests/conftest.py`: solo configuración (opciones `--nmap-target`, `--nmap-arguments` y `--nmap-ports`, más sus fixtures). No contiene tests; existe porque pytest exige declarar las opciones de línea de comandos en un `conftest.py` o en un plugin.
- `tests/test_nmap.py`: 10 tests rápidos de seguridad/validación + 1 test de integración opt-in.
- `tests/test_pdf.py`: 5 tests de generación de PDFs.

Los tests se agrupan en dos temas:

#### Grupo 1: Tests de seguridad y validación de Nmap (`tests/test_nmap.py`)

Aquí realizamos dos tipos de tests:

* **Suite rápida (`tests/test_nmap.py`)**

Probamos únicamente la lógica de *hardening*, es decir, la allowlist, targets, puertos e inyecciones. No necesitamos pasarle una red real como parámetro, ya que su objetivo es comprobar que no ejecute comandos peligrosos, pero no chequea que los resultados del escaneo sean correctos. Está pensada para ejecutarse en MLOps/CI.

* **Test de red (`tests/test_nmap.py`, marker `network`)**

Testeamos que Nmap corra, encuentre hosts y devuelva el JSON esperado. Debemos pasarle una red en la cual pueda escanear de manera real a sus hosts. Es opt-in: sin `--nmap-target` se omite.

##### Tests disponibles

*15 funciones rápidas (50 casos):*

- **Allowlist de Nmap:** `--script`, inyecciones de shell, `-oN`/`-oX`, `-iL`, `-p`, spoofing (`-D`/`-S`/`-e`), exceso de argumentos y comillas sin cerrar.
- **Targets y puertos válidos/inválidos.**
- Invocación de la tool con args maliciosos sin ejecutar Nmap (scanner monkeypatcheado a una bomba).
- Camino feliz con un `PortScanner` falso.

*1 test de integración opcional:* escaneo real de la red que le pases, con resumen de hosts encontrados.

Lo ejecutamos de la siguiente manera:

```bash
# Suite rápida: el test de red se omite
uv run pytest
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Todos los tests de Nmap (rápidos + escaneo real; default -sn)
uv run pytest tests/test_nmap.py --nmap-target 192.168.0.0/24 -s
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Solo el test de red
uv run pytest -m network --nmap-target 192.168.0.0/24 -s
```


<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 2.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Con flags permitidos y puertos
uv run pytest -m network --nmap-target 192.168.0.0/24 --nmap-arguments="-sT -F --open" --nmap-ports="22,80,443" -s
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 3.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

```bash
# Con variable de entorno
CAPAZETA_NMAP_TARGET=192.168.0.0/24 uv run pytest -m network -s
```


<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 4.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

`-s` es para ver la lista de hosts; sin él, pytest captura el `print`.

Opciones:
- `--nmap-target` RED: IP, CIDR o hostname. Sin esto, el test se omite.
- `--nmap-arguments` ARGS: default `-sn`. Se validan contra la allowlist.
- `--nmap-ports` PUERTOS: default `1-1024`. Se ignoran con `-sn` y `-F`.

El target se valida con `_validate_target` y los flags con la misma allowlist que usa el agente, así que `--nmap-arguments "--script vuln"` se rechaza igual que desde la CLI. El test no genera reportes (`save_md`/`save_pdf` quedan en `False`).

#### Grupo 2: Tests de generación de PDFs (`tests/test_pdf.py`)

Los encontramos en [`tests/test_pdf.py`](tests/test_pdf.py).

| Test | Qué cubre |
|---|---|
| `test_markdown_a_pdf_genera_pdf_valido` | `markdown_to_pdf()` con título, encabezados, listas, negrita, itálica y código. Verifica que devuelve la ruta y genera un PDF real. |
| `test_pdf_con_tabla_enlaces_y_codigo` | Renderiza una tabla Markdown con enlaces, una cita y formato inline. Ejercita `_make_table`, `_split_row`, `_is_separator` y `_inline` sin excepciones. |
| `test_save_pdf_report_sanitiza_target` | `save_pdf_report()` con target `../../etc/passwd`. Comprueba que el archivo queda dentro del `reports_dir` indicado, con nombre `nmap_etc_passwd_<ts>.pdf`, sin `/` ni `..`, y que es un PDF válido. |
| `test_save_general_pdf_report_usa_prefix` | `save_general_pdf_report()` con prefijo `"informe wazuh"` → `informe_wazuh_<ts>.pdf`. |
| `test_pdf_con_contenido_vacio_o_html` (2 casos) | Contenido vacío → fallback `"Sin contenido."`; contenido con `<script>` y `&` → el `escape()` evita que ReportLab reviente al parsear el XML interno de `Paragraph`. |

`assert_pdf()` valida tres cosas concretas:
- el archivo existe,
- empieza con la firma `%PDF-`,
- termina en `%%EOF` y pesa más de 500 bytes.

*Qué no prueba*
- No compara el texto interno ni la cantidad de páginas (no se agregó pypdf ni dependencias extra).
- No valida la apariencia visual (márgenes, colores, fuentes).
- No prueba los wrappers de Wazuh/MITRE, ni `run_nmap_scan(save_pdf=True)` de punta a punta; prueba las funciones de `pdf_report.py` directamente.
- No escribe en `reports/`: todo va a `tmp_path`.

Probamos solo esos tests de la siguiente manera:

```bash
uv run pytest tests/test_pdf.py
```

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 5.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>


### Integracion continua

Vamos a subir el workflow de testing a Git Hub, asi se ejecutan los tests correspondientes cuando se realiza cada push. Vamos a testear solamente la integridad de comandos de nmap y el report de pdf, ya que para no podemos usar una network real al ejecutar la prueba en github, por lo que este quedara afuera

Se agrego el siguiente `test.yml` dentro de la carpeta de `.github/workflow`
```bash
name: Tests

on:
  push:
    branches:
      - master
      - "features/**"
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

concurrency:
  group: tests-${{ github.ref }}
  cancel-in-progress: true

jobs:
  pytest:
    name: pytest (suite rápida)
    runs-on: ubuntu-latest
    timeout-minutes: 15
    steps:
      - name: Checkout
        uses: actions/checkout@v5

      - name: Instalar uv y Python 3.10
        uses: astral-sh/setup-uv@v10
        with:
          python-version: "3.10"
          enable-cache: true

      - name: Instalar dependencias
        run: uv sync --locked --group dev

      - name: Correr tests sin red
        run: uv run --no-sync pytest -m "not network"
```
Una vez realizado del commit y el push correspondientes vamos a git hub para ver si se ejecutaron los tests al pushear


<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 6.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Vemos que se ejecutaron y superaron los tests correctamente

<div align="center">
  <img src="./img/06_mejoras_trabajo_final/image copy 7.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>